# Financial Fraud Detection with Blind Insight Data

This notebook demonstrates how to use financial fraud data from Blind Insight for machine learning tasks with scikit-learn.

## Prerequisites

1. Install required packages:
   ```bash
   pip install -r requirements.txt
   ```

2. Ensure the backend API is running:
   ```bash
   cd cube-server
   node index.js
   ```

3. Make sure you have uploaded the financial fraud dataset to Blind Insight using the importer tool.


## Setup and Imports


In [1]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient, load_fraud_from_blind

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

## Load Data from Blind Insight

Instead of using standard datasets, we load financial fraud data from Blind Insight.

**Note**: Update the organization, dataset_slug, and schema_slug to match your Blind Insight setup.


### Workflows
- **Encrypted-only (recommended)**: Use Blind Insight encrypted primitives (count, avg, min, max, ranges) and keep rows encrypted. See the *Encrypted Aggregation Classifier* section below.
- **Plaintext ML (illustrative only)**: Decrypt data and run scikit-learn directly. This is not encrypted and is kept for reference/testing.


In [2]:
# Configuration - Plaintext ML example (illustrative)
# NOTE: This decrypts data; for encrypted-only workflows see the section below.
ORGANIZATION = "demo"
DATASET_SLUG = "FraudAnalysis"
SCHEMA_SLUG = "FraudAnalysis"
API_URL = "https://proxy.local.blindinsight.io/"

# Authentication for backend API (required for metadata lookups like schema ID resolution)
# If you get 403 errors, provide your Blind Insight credentials here:
USERNAME = "data_owner@localhost"  # Your Blind Insight account email
PASSWORD = "blindinsight"  # Your Blind Insight account password

# Schema ID (for encrypted datasets, provide this directly to avoid metadata API calls)
SCHEMA_ID = "oKVxDa56fi4EwrgjpNiibw"  # Schema ID for the encrypted fraud dataset

# Load raw data as DataFrame so we can process categorical columns
# Available columns: amount, country, device_risk_score, hour, ip_risk_score, 
# is_fraud, merchant_category, transaction_id, transaction_type, user_id
# 
# NOTE: We exclude device_risk_score and ip_risk_score as they make the problem
# trivial (100% accuracy). Instead we use amount, hour, and categorical features.

client = BlindInsightClient(
    api_url=API_URL,
    username=USERNAME,
    password=PASSWORD,
    verify_ssl=False
)

# Load full dataset as DataFrame
df_raw = client.load_data(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    limit=10000,  # Load enough data to have 500 fraud cases
    decrypt=True,
    schema_id=SCHEMA_ID
)

# Handle nested 'data' column if present (common in Blind Insight responses)
import pandas as pd
if 'data' in df_raw.columns and df_raw['data'].dtype == object:
    sample_val = df_raw['data'].iloc[0] if len(df_raw) > 0 else None
    if isinstance(sample_val, dict):
        # Expand nested dictionaries into separate columns
        data_dicts = df_raw['data'].apply(lambda x: x if isinstance(x, dict) else {})
        df = pd.json_normalize(data_dicts)
        print(f"Expanded DataFrame columns: {df.columns.tolist()}")
    else:
        df = df_raw
else:
    df = df_raw

print(f"\nLoaded {len(df)} records")
print(f"Available columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['is_fraud'].value_counts())
print(f"\nNote: Binary classification - 0 = not fraud, 1 = fraud")


Expanded DataFrame columns: ['amount', 'country', 'device_risk_score', 'hour', 'ip_risk_score', 'is_fraud', 'merchant_category', 'transaction_id', 'transaction_type', 'user_id']

Loaded 9880 records
Available columns: ['amount', 'country', 'device_risk_score', 'hour', 'ip_risk_score', 'is_fraud', 'merchant_category', 'transaction_id', 'transaction_type', 'user_id']

Class distribution:
is_fraud
0    9386
1     494
Name: count, dtype: int64

Note: Binary classification - 0 = not fraud, 1 = fraud


## Prepare Data for Classification

For fraud detection, we'll use:
- **Numeric features**: `amount` (scaled to cents as integer: multiply by 100), `hour`
- **Categorical features**: `transaction_type`, `merchant_category`, `country` (one-hot encoded)

We exclude `device_risk_score` and `ip_risk_score` as they make the problem trivial (100% accuracy).

**Note**: Amount is scaled by 100 (e.g., $63.99 becomes 6399 cents) so encrypted aggregations work (proxy doesn't support float aggregations).

The target variable `is_fraud` is binary (0 = not fraud, 1 = fraud).


In [3]:
# Preprocessing: 
# 1. Scale amounts to cents (multiply by 100) for integer aggregation support
# 2. One-hot encode categorical features
# 3. Combine numeric and encoded features

from sklearn.preprocessing import OneHotEncoder

# Define feature columns
numeric_cols = ["amount", "hour"]
categorical_cols = ["transaction_type", "merchant_category", "country"]
target_col = "is_fraud"

# Scale amount to cents (multiply by 100, round to integer)
# This allows encrypted aggregations to work (proxy doesn't support float aggregations)
df["amount"] = (df["amount"] * 100).round(0).astype(int)

# Extract numeric features
X_numeric = df[numeric_cols].values

# One-hot encode categorical features
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_categorical = encoder.fit_transform(df[categorical_cols])

# Get feature names for reference
categorical_feature_names = encoder.get_feature_names_out(categorical_cols)
all_feature_names = list(numeric_cols) + list(categorical_feature_names)

# Combine numeric and categorical features
X = np.hstack([X_numeric, X_categorical])

# Extract target
y = df[target_col].values.astype(int)

print(f"Feature matrix shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nNumeric features: {numeric_cols}")
print(f"Categorical features (one-hot encoded): {categorical_cols}")
print(f"Total features: {len(all_feature_names)}")
print(f"\nFeature names: {all_feature_names}")
print(f"\nClass distribution: {np.bincount(y)}")


Feature matrix shape: (9880, 17)
Target shape: (9880,)

Numeric features: ['amount', 'hour']
Categorical features (one-hot encoded): ['transaction_type', 'merchant_category', 'country']
Total features: 17

Feature names: ['amount', 'hour', 'transaction_type_ATM', 'transaction_type_Online', 'transaction_type_POS', 'transaction_type_QR', 'merchant_category_Clothing', 'merchant_category_Electronics', 'merchant_category_Food', 'merchant_category_Grocery', 'merchant_category_Travel', 'country_DE', 'country_FR', 'country_NG', 'country_TR', 'country_UK', 'country_US']

Class distribution: [9386  494]


In [4]:
# Show sample data and statistics
print("Sample data (first 5 rows):")
print(f"Amount (cents): {X[:5, 0]}")
print(f"Hour: {X[:5, 1]}")
print(f"One-hot encoded features (first 5): {X[:5, 2:5]}...")
print(f"\nSample targets: {y[:5]}")

print(f"\nNumeric feature statistics:")
print(f"Amount (cents) - Min: {X[:, 0].min():.0f}, Max: {X[:, 0].max():.0f}, Mean: {X[:, 0].mean():.0f}")
print(f"Hour - Min: {X[:, 1].min():.0f}, Max: {X[:, 1].max():.0f}, Mean: {X[:, 1].mean():.1f}")

print(f"\nCategorical value counts:")
for col in categorical_cols:
    print(f"  {col}: {df[col].nunique()} unique values")


Sample data (first 5 rows):
Amount (cents): [  6399.  11331.  13196. 428418.   7670.]
Hour: [12.  9. 18. 12.  7.]
One-hot encoded features (first 5): [[0. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [1. 0. 0.]
 [0. 0. 1.]]...

Sample targets: [0 0 0 1 0]

Numeric feature statistics:
Amount (cents) - Min: 100, Max: 1162821, Mean: 17786
Hour - Min: 0, Max: 23, Mean: 14.2

Categorical value counts:
  transaction_type: 4 unique values
  merchant_category: 5 unique values
  country: 6 unique values


## Create Balanced Train/Test Split

To avoid class imbalance issues in ML, we create a balanced training set with:
- 250 fraud cases
- 250 non-fraud cases

The remaining data is used for testing (includes the rest of fraud and non-fraud cases).


In [5]:
# Create balanced train/test split
# Training: 250 fraud + 250 non-fraud
# Testing: remaining samples

# Separate indices by class
fraud_idx = np.where(y == 1)[0]
non_fraud_idx = np.where(y == 0)[0]

print(f"Total fraud cases: {len(fraud_idx)}")
print(f"Total non-fraud cases: {len(non_fraud_idx)}")

# Shuffle indices for random sampling
np.random.seed(42)  # Reproducibility
np.random.shuffle(fraud_idx)
np.random.shuffle(non_fraud_idx)

# Training set: 250 fraud + 250 non-fraud
train_fraud = fraud_idx[:250]
train_non_fraud = non_fraud_idx[:250]
train_idx = np.concatenate([train_fraud, train_non_fraud])

# Test set: remaining samples
test_fraud = fraud_idx[250:]
test_non_fraud = non_fraud_idx[250:]
test_idx = np.concatenate([test_fraud, test_non_fraud])

# Shuffle training indices so fraud/non-fraud are mixed
np.random.shuffle(train_idx)

# Split data
X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"\nTraining set: {len(X_train)} samples")
print(f"  - Fraud: {np.sum(y_train == 1)}")
print(f"  - Non-fraud: {np.sum(y_train == 0)}")
print(f"\nTest set: {len(X_test)} samples")
print(f"  - Fraud: {np.sum(y_test == 1)}")
print(f"  - Non-fraud: {np.sum(y_test == 0)}")


Total fraud cases: 494
Total non-fraud cases: 9386

Training set: 500 samples
  - Fraud: 250
  - Non-fraud: 250

Test set: 9380 samples
  - Fraud: 244
  - Non-fraud: 9136


## Train Logistic Regression Model

Train a logistic regression classifier on the balanced training set.


In [6]:
# Train logistic regression on balanced training set
logreg = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs')
clf = logreg.fit(X_train, y_train)

print("Model trained successfully!")
print(f"Model: {clf}")
print(f"\nTraining accuracy: {accuracy_score(y_train, clf.predict(X_train)):.4f}")


Model trained successfully!
Model: LogisticRegression(max_iter=1000)

Training accuracy: 0.8900


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:333: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:333: RuntimeWarning: overflow 

## Evaluate Model on Test Set

Evaluate the trained model on the held-out test set to see real-world performance.

In [7]:
# Evaluate on test set
y_pred = clf.predict(X_test)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Non-Fraud', 'Fraud']))

print(f"\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nInterpretation:")
print(f"  True Negatives (correct non-fraud): {cm[0, 0]}")
print(f"  False Positives (non-fraud predicted as fraud): {cm[0, 1]}")
print(f"  False Negatives (fraud predicted as non-fraud): {cm[1, 0]}")
print(f"  True Positives (correct fraud): {cm[1, 1]}")

Test Accuracy: 0.9854 (98.54%)

Classification Report:
              precision    recall  f1-score   support

   Non-Fraud       0.99      0.99      0.99      9136
       Fraud       0.69      0.80      0.74       244

    accuracy                           0.99      9380
   macro avg       0.84      0.89      0.87      9380
weighted avg       0.99      0.99      0.99      9380


Confusion Matrix:
[[9048   88]
 [  49  195]]

Interpretation:
  True Negatives (correct non-fraud): 9048
  False Positives (non-fraud predicted as fraud): 88
  False Negatives (fraud predicted as non-fraud): 49
  True Positives (correct fraud): 195


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [8]:
# Example: Use encrypted filters to narrow dataset before decryption
# Encrypted aggregation-based classifier (no decryption)
# Binary fraud detection using amount feature
# 
# NOTE: The proxy doesn't support count aggregations on float fields like "amount".
# Workaround: Use range filters with query() to count records, or use avg/min/max aggregations.
# Alternative: Use integer fields or integer-scaled values for count aggregations.

from blind_insight_client import BlindInsightClient
import requests
from urllib.parse import urlencode

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
feature = "amount"  # Use transaction amount as the feature
class_a = "0"  # Not fraud (is_fraud = 0)
class_b = "1"  # Fraud (is_fraud = 1)

# Helper to extract aggregation value (supports both shapes)
# Shape A: {records: [{data: {value: X}}]}
# Shape B: {records: [{value: X, aggregation_type: ...}]}

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    # Shape A
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    # Shape B
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# WORKAROUND: Since count() doesn't work on float fields via proxy, we use a different approach:
# 1. Use query() with range filters to get record counts
# 2. Or use avg/min/max aggregations instead of count
# 3. Or use the backend API directly (requires hashing, more complex)

# Method 1: Use query() with range filters to count records
# This works because query() supports range filters on floats, we just count the results
def count_records_with_filter(extra_filters, amount_range=None):
    """Count records by querying with filters and counting results."""
    filters = extra_filters.copy() if extra_filters else []
    if amount_range:
        filters.append(f"{feature}:{amount_range}")
    
    result = client.query(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        limit=10000,  # Large limit to get all matching records
        offset=0,
        filters=filters,
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    records = result.get("records", [])
    return len(records)

# Precompute totals per class using query() instead of count aggregation
print("Counting records per class using query() method (workaround for float count limitation)...")
count_a_total = count_records_with_filter([f"is_fraud:{class_a}"])
count_b_total = count_records_with_filter([f"is_fraud:{class_b}"])

print(f"Total class A (not fraud): {count_a_total}")
print(f"Total class B (fraud): {count_b_total}")

# Thresholds over transaction amount
# Adjust range based on your data - example: 0 to 10000 with step 500
thresholds = range(0, 10001, 500)  # Adjust based on your amount range

best_acc = -1
best_t = None
best_counts = None

print(f"\nSearching {len(thresholds)} thresholds...")
for t in thresholds:
    # Count records with amount < threshold for each class
    count_a_left = count_records_with_filter(
        [f"is_fraud:{class_a}"],
        amount_range=f"<{t}"
    )
    count_b_left = count_records_with_filter(
        [f"is_fraud:{class_b}"],
        amount_range=f"<{t}"
    )

    # Accuracy: predict class_a if amount < threshold, else class_b
    correct = count_a_left + (count_b_total - count_b_left)
    total = count_a_total + count_b_total
    acc = correct / total if total else 0.0

    if acc > best_acc:
        best_acc = acc
        best_t = t
        best_counts = (count_a_left, count_b_left, correct, total)

print(f"\nBest threshold on encrypted counts: {best_t:.3f}")
print(f"Encrypted-rule accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Counts at best threshold -> class_a_left: {best_counts[0]:.0f}, class_b_left: {best_counts[1]:.0f}, correct: {best_counts[2]:.0f}/{best_counts[3]:.0f}")

print("\nRule: predict not fraud (0) if amount < threshold, else fraud (1) (encrypted counts only)")
print("\nNote: This uses query() with range filters instead of count() aggregation due to proxy limitation on float fields.")


Counting records per class using query() method (workaround for float count limitation)...
Total class A (not fraud): 10000
Total class B (fraud): 10000

Searching 21 thresholds...

Best threshold on encrypted counts: 0.000
Encrypted-rule accuracy: 0.5000 (50.00%)
Counts at best threshold -> class_a_left: 10000, class_b_left: 10000, correct: 10000/20000

Rule: predict not fraud (0) if amount < threshold, else fraud (1) (encrypted counts only)

Note: This uses query() with range filters instead of count() aggregation due to proxy limitation on float fields.


## Alternative: Using the Client Directly

You can also use the client directly for more control over the data loading process.


## Encrypted Aggregation Classifier (no decryption)

This section demonstrates classification using only Blind Insight's encrypted primitives:
- Use encrypted filters and aggregations (count/min/max/ranges) on encrypted data
- Never decrypt rows
- Build a simple rule-based classifier using aggregate stats only

**Dataset note**: For this demo we use transaction amount as the feature for threshold-based classification.

Approach (fraud vs non-fraud using amount threshold):
1. Search thresholds over transaction amount using encrypted `count(<t)` per class.
2. Pick the threshold with best accuracy on encrypted counts.
3. No row-level decryption; only aggregates are returned.


## Encrypted averages: transaction amount per fraud class
Use Blind Insight encrypted `avg` to compute the mean transaction amount for fraud vs non-fraud transactions.

**Note**: For encrypted aggregations to work, amounts must be stored as integers (cents) in Blind Insight. If using floats, you'll get a 422 error. The importer should scale amounts by 100 (e.g., $63.99 -> 6399 cents) before storing.


In [9]:
# Encrypted avg of amount per fraud class
# Compute the mean transaction amount for fraud vs non-fraud transactions
# NOTE: Amount must be stored as integers (cents) in Blind Insight for aggregations to work.
# If data is stored as floats, this will fail with 422 error.
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
feature = "amount"  # Transaction amount field (should be stored as cents/integers)
classes = ["0", "1"]  # is_fraud: 0 = not fraud, 1 = fraud
class_labels = {"0": "Not Fraud", "1": "Fraud"}

# Helper to extract aggregation value (supports both shapes)
def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# Range for amount in cents: 0 to 10,000,000 cents ($100,000)
# If your data is still stored as floats, this will fail - re-import data with integer amounts.

for cls in classes:
    try:
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~10000000)",  # Range for amount in cents
            extra_filters=[f"is_fraud:{cls}"],
            decrypt=False,  # keep encrypted; avg is computed inside Blind Insight
            schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
        )
        mean_val = agg_value(resp)
        # Convert back to dollars for display
        print(f"Mean {feature} for {class_labels[cls]} (is_fraud={cls}): {mean_val/100:.2f} (cents: {mean_val:.0f})")
    except ValueError as e:
        if "aggregation operations are not supported for float values" in str(e) or "422" in str(e):
            print(f"Error: Aggregation failed. Amount field may still be stored as floats.")
            print(f"  To fix: Re-import the data with amounts scaled to cents (multiply by 100).")
            print(f"  Error details: {str(e)[:150]}")
        else:
            raise



Error: Aggregation failed. Amount field may still be stored as floats.
  To fix: Re-import the data with amounts scaled to cents (multiply by 100).
  Error details: Aggregation query failed (422 Unprocessable Entity). This might mean:
1. The aggregation expression format is not supported: amount:avg(0~10000000)
2.
Error: Aggregation failed. Amount field may still be stored as floats.
  To fix: Re-import the data with amounts scaled to cents (multiply by 100).
  Error details: Aggregation query failed (422 Unprocessable Entity). This might mean:
1. The aggregation expression format is not supported: amount:avg(0~10000000)
2.


## Encrypted averages per batch (example)
Compute means for numeric features in batches using encrypted `avg`. 

**Note**: This example uses `dataset-order` which may not exist in all datasets. Numeric fields must be stored as integers for aggregations to work. This cell is provided as an example that may need adaptation for your specific dataset.


In [10]:
# Batch means over all rows, in batches
# NOTE: This cell requires a "dataset-order" field which may not exist in your dataset.
# This is an advanced example that demonstrates batch processing. You may need to skip this cell
# if your dataset doesn't have a dataset-order field, or adapt it to use a different ordering field.

# Make sure you've run the data loading cell (cell 5) first to define ORGANIZATION, DATASET_SLUG, 
# SCHEMA_SLUG, API_URL, USERNAME, PASSWORD, and SCHEMA_ID variables
from blind_insight_client import BlindInsightClient

# Verify required variables are defined (check in globals() to avoid NameError)
required_vars = ['ORGANIZATION', 'DATASET_SLUG', 'SCHEMA_SLUG', 'API_URL', 'USERNAME', 'PASSWORD', 'SCHEMA_ID']
missing_vars = [var for var in required_vars if var not in globals() or not globals()[var]]
if missing_vars:
    raise ValueError(
        f"The following variables are not defined: {', '.join(missing_vars)}. "
        f"Please run the data loading cell (cell 5) first to define these variables."
    )

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
# Use fraud dataset numeric fields - amount should be stored as cents (integers)
features = ["amount", "hour"]  # Fraud dataset numeric fields (excluding risk scores)
batch_size = 10

print("Note: This cell requires 'dataset-order' field and integer-stored numeric fields.")
print("If you get errors, ensure data is imported with amounts as cents (multiply by 100).")
print("You can skip this cell or adapt it for your specific dataset.\n")

# Reuse agg_value if defined; otherwise define here

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# This cell demonstrates batch processing using dataset-order field
# NOTE: The fraud dataset may not have a "dataset-order" field, so this cell may not work.
# This is an advanced example that requires a dataset with an ordering field.

try:
    # Determine total rows via encrypted count on dataset-order
    # NOTE: This will fail if dataset-order field doesn't exist in your schema
    TOTAL_MAX = 100000  # a large upper bound
    count_resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"dataset-order:count(0~{TOTAL_MAX})",
        decrypt=False,
        schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
    )
    total_rows = int(agg_value(count_resp))
    print(f"Total rows detected: {total_rows}")

    start = 0
    while start < total_rows:
        end = min(start + batch_size - 1, total_rows - 1)
        batch_filter = f"dataset-order:{start}~{end}"
        print(f"\nBatch {start}-{end}:")
        for feature in features:
            try:
                # Range: amount in cents (0~10M = $100k), hour (0~24)
                range_max = 10000000 if feature == "amount" else 24
                resp = client.aggregate(
                    organization=ORGANIZATION,
                    dataset_slug=DATASET_SLUG,
                    schema_slug=SCHEMA_SLUG,
                    agg_filter=f"{feature}:avg(0~{range_max})",
                    extra_filters=[batch_filter],
                    decrypt=False,
                    schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
                )
                mean_val = agg_value(resp)
                if feature == "amount":
                    print(f"  mean {feature}: ${mean_val/100:.2f} ({mean_val:.0f} cents)")
                else:
                    print(f"  mean {feature}: {mean_val:.1f}")
            except ValueError as e:
                if "422" in str(e) or "aggregation operations are not supported" in str(e):
                    print(f"  Error: avg() on '{feature}' failed - ensure data is stored as integers")
                else:
                    raise
        start += batch_size
except ValueError as e:
    if "label not found" in str(e) or "dataset-order" in str(e):
        print("=" * 60)
        print("SKIPPED: This cell requires a 'dataset-order' field which doesn't exist")
        print("in the fraud dataset. This is an advanced example for datasets that")
        print("have an ordering field.")
        print("=" * 60)
        print(f"\nError details: {str(e)[:200]}")
    else:
        raise



Note: This cell requires 'dataset-order' field and integer-stored numeric fields.
If you get errors, ensure data is imported with amounts as cents (multiply by 100).
You can skip this cell or adapt it for your specific dataset.

SKIPPED: This cell requires a 'dataset-order' field which doesn't exist
in the fraud dataset. This is an advanced example for datasets that
have an ordering field.

Error details: Aggregation query failed (422 Unprocessable Entity). This might mean:
1. The aggregation expression format is not supported: dataset-order:count(0~100000)
2. The proxy might not be configured to handl


In [11]:
# Example: Load data as a pandas DataFrame for exploration
# Make sure you've run the data loading cell (cell 5) first to define required variables
from blind_insight_client import BlindInsightClient

# Verify required variables are defined
required_vars = ['ORGANIZATION', 'DATASET_SLUG', 'SCHEMA_SLUG', 'API_URL', 'USERNAME', 'PASSWORD', 'SCHEMA_ID']
missing_vars = [var for var in required_vars if var not in globals() or not globals()[var]]
if missing_vars:
    raise ValueError(
        f"The following variables are not defined: {', '.join(missing_vars)}. "
        f"Please run the data loading cell (cell 5) first to define these variables."
    )

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)

# Check API health
health = client.health_check()
print(f"API Status: {health}")

# Load full dataset as DataFrame
# Pass schema_id to avoid metadata lookup that requires authentication
df = client.load_data(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    limit=150,
    schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
)

print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataFrame info:")
print(df.info())


API Status: {'status': 'ok', 'api_version': '10.8.6', 'behind_proxy': True}

DataFrame shape: (150, 4)

Column names: ['data', 'id', 'schema', 'url']

First few rows:
                                                data                      id  \
0  {'amount': 63.99286999198463, 'country': 'FR',...  bVAmQ4kTHV68Eqk7beEF2o   
1  {'amount': 113.30575124310727, 'country': 'TR'...  LfJn5wf8jii5XAH9YHvSzL   
2  {'amount': 131.96349737388977, 'country': 'FR'...  3M3HdVPtxSNFCfg4KvV7HJ   
3  {'amount': 4284.177966789535, 'country': 'TR',...  F9HgpSLQbqG7rnyWtehtif   
4  {'amount': 76.69547978824451, 'country': 'TR',...  TKaJVc7yCaVM8ghjrLP6JN   

                                              schema  \
0  https://localhost:8080/api/schemas/oKVxDa56fi4...   
1  https://localhost:8080/api/schemas/oKVxDa56fi4...   
2  https://localhost:8080/api/schemas/oKVxDa56fi4...   
3  https://localhost:8080/api/schemas/oKVxDa56fi4...   
4  https://localhost:8080/api/schemas/oKVxDa56fi4...   

               

## Summary

This notebook demonstrates:

1. Loading financial fraud data from Blind Insight
2. Preprocessing data with one-hot encoding for categorical features
3. Creating balanced train/test splits for ML
4. Training a logistic regression classifier
5. Evaluating model accuracy on held-out test data

The key difference from standard scikit-learn examples is that instead of loading local files, we use:
```python
X, y = load_fraud_from_blind(organization, dataset_slug, schema_slug)
```

This allows data scientists to work with encrypted, privacy-preserving data stored in Blind Insight while using standard ML libraries and workflows.
